In [1]:
import xarray as xr
import numpy as np
import pandas as pd

from snobedo.snotel import SnotelLocations
from snobedo.snotel import SnotelLocations, CsvParser

from snobedo.lib.dask_utils import start_cluster, client_ip_and_port

from pathlib import Path, PurePath

import matplotlib.pyplot as plt

import os
import glob

In [2]:
# client = start_cluster(4, 24)
# client_ip_and_port(client)

In [3]:
SHARED_STORE = PurePath('/uufs/chpc.utah.edu/common/home/skiles-group1')
DATA_DIR = SHARED_STORE.joinpath('jmeyer')
SNOTEL_DIR = DATA_DIR.joinpath( 'Snotel')
from pathlib import Path, PurePath
snotel_sites = SnotelLocations()
snotel_sites.load_from_json(SNOTEL_DIR / 'site-locations/snotel_sites.json')
snotel_sites.Irwin

SnotelSite(name='Irwin', lon=[317131], lat=[4306375])

In [4]:
from dask.distributed import Client
from dask_jobqueue import SLURMCluster

cluster = SLURMCluster(
    cores=8,
    processes=4,
    memory="32GB",
    account="notchpeak-shared-short", # project
    queue="notchpeak-shared-short",
    walltime="2:00:00",
)

# Scale the cluster — adjust this based on your workload
cluster.scale(jobs=2)  # e.g., 2 jobs → 2 × 8 = 16 cores total

client = Client(cluster)  # Attach Dask client to the cluster

In [4]:
# from dask.distributed import Client

# # Optional: start Dask client to monitor computation
# client = Client()

# # file_pattern = '/uufs/chpc.utah.edu/common/home/u1037042/olson/snow-data/hrrr_toposplit/toposplit_*.nc'
# file_pattern = '/uufs/chpc.utah.edu/common/home/u1037042/olson/snow-data/hrrr_toposplit_2022/toposplit_*.nc'
# # file_pattern = '/uufs/chpc.utah.edu/common/home/u1037042/olson/snow-data/hrrr_toposplit_2023/toposplit_*.nc'

# # file_pattern = f"{SHARED_STORE_RUN}/erw_spires_dsw3_dlwrf/{water_year}/erw_50m/run*/net_solar.nc"

# HRRR_solar = xr.open_mfdataset(
#     file_pattern,
#     combine='by_coords',
#     parallel=True, chunks={'time': 24}, # 'y' :10, 'x': 10},
# )

In [5]:
# from dask.distributed import Client

# # Optional: start Dask client to monitor computation
# client = Client()

file_pattern = '/uufs/chpc.utah.edu/common/home/u1037042/olson/snow-data/hrrr_toposplit_new/hrrr_toposplit_2022/toposplit_*.nc'
# file_pattern = '/uufs/chpc.utah.edu/common/home/u1037042/olson/snow-data/hrrr_toposplit_new/hrrr_toposplit_2023/toposplit_*.nc'

# file_pattern = f"{SHARED_STORE_RUN}/erw_spires_dsw3_dlwrf/{water_year}/erw_50m/run*/net_solar.nc"

HRRR_solar = xr.open_mfdataset(
    file_pattern,
    combine='by_coords',
    parallel=True, chunks={'time': 24}, # 'y' :10, 'x': 10},
)

In [9]:
# client.shutdown()

In [8]:
# SOS
sos_path = '/uufs/chpc.utah.edu/common/home/u1037042/olson/snow-data/station-data/sos-isfs/hourlyAvg_radiation_uw.nc'
# sos_single = '/uufs/chpc.utah.edu/common/home/u1037042/olson/snow-data/station-data/sos-isfs/sos_isfs_qc_geo_tiltcor_5min_v20241227/isfs_sos_qc_geo_tiltcor_5min_v2_20230212.nc'
ds = xr.open_dataset(sos_path)

In [9]:
sos_coords = {"lat": ds.latitude_uw.isel(time=0).values.item(), "lon": ds.longitude_uw.isel(time=0).values.item()}

print(sos_coords)

{'lat': 38.941712, 'lon': -106.97328799999998}


In [6]:
#SOS coords
{'lat': 38.941712, 'lon': 38.941712}

{'lat': 38.941712, 'lon': 38.941712}

In [10]:
import geopandas as gpd
from shapely.geometry import Point

def convert_to_utm(coords):
    """
    Convert coordinates from WGS84 (lat/lon) to UTM Zone 13N (EPSG:26913)
    
    :param coords: Dictionary with 'lat' and 'lon' keys
    :return: Dictionary with 'utm_x' and 'utm_y' keys for UTM coordinates
    """
    # Create a GeoDataFrame with the coordinates
    gdf = gpd.GeoDataFrame(
        {'geometry': [Point(coords['lon'], coords['lat'])]}, 
        crs="EPSG:4326"  # WGS84 (Lat/Lon)
    )

    # Convert to UTM Zone 13N
    gdf_utm = gdf.to_crs(epsg=26913)

    # Extract the UTM X and Y coordinates
    utm_x, utm_y = gdf_utm.geometry.x[0], gdf_utm.geometry.y[0]
    
    # Return the result as a dictionary
    return {'utm_x': utm_x, 'utm_y': utm_y}

# Define m1_coords and s3_coords (example data)
m1_coords = {"lat": 38.95615768432617, "lon": -106.98785400390625}
# s3_coords = {"lat": 39.750715, "lon": -104.998649}
s3_coords = {"lat": 38.941555, "lon": -106.97313}
sos_coords = {"lat": 38.941712, "lon": -106.97328799999998}

# Convert both coordinates
m1_utm = convert_to_utm(m1_coords)
s3_utm = convert_to_utm(s3_coords)
sos_utm = convert_to_utm(sos_coords)

# Combine the results into a final object
final_utm_coords = {
    'm1_coords_utm': m1_utm, #ARM QCRAD also at M1 site! (CHECK)
    's3_coords_utm': s3_utm,
    'sos_coords_utm': sos_utm
}

# Print the result
print(final_utm_coords)


{'m1_coords_utm': {'utm_x': np.float64(327754.55108678306), 'utm_y': np.float64(4313790.299013529)}, 's3_coords_utm': {'utm_x': np.float64(328995.37306967156), 'utm_y': np.float64(4312141.891803242)}, 'sos_coords_utm': {'utm_x': np.float64(328982.05598244135), 'utm_y': np.float64(4312159.612842733)}}


In [12]:
# Subset once using .sel with method='nearest'
irwin_data = HRRR_solar.sel(x=snotel_sites.Irwin.lon, y=snotel_sites.Irwin.lat, method='nearest')

# Drop unnecessary dimensions
irwin_data = irwin_data.squeeze(['x', 'y'])

# Now compute all at once
irwin_data = irwin_data[['ghi', 'dsw1', 'dsw3', 'dsw3h', 'k']].compute()

# Access variables
irwin_ghi = irwin_data['ghi']
irwin_dsw1 = irwin_data['dsw1']
irwin_dsw30 = irwin_data['dsw3']   # Assuming `dsw3` is the same as `dsw30`
irwin_dsw3 = irwin_data['dsw3h']
irwin_k = irwin_data['k']

irwin_dir = irwin_data['dir']
irwin_dif = irwin_data['dif']

KeyboardInterrupt: 

In [ ]:
# Define helper function to extract and compute data at one site
def extract_site_data(site_key):
    x = final_utm_coords[site_key]['utm_x']
    y = final_utm_coords[site_key]['utm_y']
    
    site_ds = HRRR_solar.sel(x=x, y=y, method='nearest')[['ghi', 'dsw1', 'dsw3', 'dsw3h', 'k']]
    return site_ds.squeeze().compute()

# Extract for M1
m1_data = extract_site_data('m1_coords_utm')
m1_ghi = m1_data['ghi']
m1_dsw1 = m1_data['dsw1']
m1_dsw30 = m1_data['dsw3']
m1_dsw3 = m1_data['dsw3h']
m1_k = m1_data['k']
m1_dir = m1_data['dir']
m1_dif = m1_data['dif']

# Extract for S3
s3_data = extract_site_data('s3_coords_utm')
s3_ghi = s3_data['ghi']
s3_dsw1 = s3_data['dsw1']
s3_dsw30 = s3_data['dsw3']
s3_dsw3 = s3_data['dsw3h']
s3_k = s3_data['k']


# Extract for SOS # 2023 only
sos_data = extract_site_data('sos_coords_utm')
sos_ghi = sos_data['ghi']
sos_dsw1 = sos_data['dsw1']
sos_dsw30 = sos_data['dsw3']
sos_dsw3 = sos_data['dsw3h']
sos_k = sos_data['k']

In [10]:
import os

# Create folder if it doesn't already exist
save_dir = 'new-station-data-22'
os.makedirs(save_dir, exist_ok=True)

# S3 datasets
# s3_obs.to_netcdf(os.path.join(save_dir, 's3_obs.nc'))
# s3_sail.to_netcdf(os.path.join(save_dir, 's3_sail.nc'))
s3_dsw3.to_netcdf(os.path.join(save_dir, 's3_dsw3.nc'))
s3_k.to_netcdf(os.path.join(save_dir, 's3_k.nc'))
s3_dsw1.to_netcdf(os.path.join(save_dir, 's3_dsw1.nc'))
s3_ghi.to_netcdf(os.path.join(save_dir, 's3_ghi.nc'))
s3_dsw30.to_netcdf(os.path.join(save_dir, 's3_dsw30.nc'))
# s3_dir.to_netcdf(os.path.join(save_dir, 's3_dir.nc'))
# s3_dif.to_netcdf(os.path.join(save_dir, 's3_dif.nc'))

# M1 datasets
# m1_obs.to_netcdf(os.path.join(save_dir, 'm1_obs.nc'))
# m1_sail.to_netcdf(os.path.join(save_dir, 'm1_sail.nc'))
m1_dsw3.to_netcdf(os.path.join(save_dir, 'm1_dsw3.nc'))
m1_k.to_netcdf(os.path.join(save_dir, 'm1_k.nc'))
m1_dsw1.to_netcdf(os.path.join(save_dir, 'm1_dsw1.nc'))
m1_ghi.to_netcdf(os.path.join(save_dir, 'm1_ghi.nc'))
m1_dsw30.to_netcdf(os.path.join(save_dir, 'm1_dsw30.nc'))
# m1_dir.to_netcdf(os.path.join(save_dir, 'm1_dir.nc'))
# # m1_dif.to_netcdf(os.path.join(save_dir, 'm1_dif.nc'))

# Irwin datasets
irwin_dsw3.to_netcdf(os.path.join(save_dir, 'irwin_dsw3.nc'))
irwin_k.to_netcdf(os.path.join(save_dir, 'irwin_k.nc'))
irwin_dsw1.to_netcdf(os.path.join(save_dir, 'irwin_dsw1.nc'))
irwin_ghi.to_netcdf(os.path.join(save_dir, 'irwin_ghi.nc'))
irwin_dsw30.to_netcdf(os.path.join(save_dir, 'irwin_dsw30.nc'))
# irwin_dir.to_netcdf(os.path.join(save_dir, 'irwin_dir.nc'))
# irwin_dif.to_netcdf(os.path.join(save_dir, 'irwin_dif.nc'))

# SOS - ONLY 2023
sos_dsw3.to_netcdf(os.path.join(save_dir, 'sos_dsw3.nc'))
sos_k.to_netcdf(os.path.join(save_dir, 'sos_k.nc'))
sos_dsw1.to_netcdf(os.path.join(save_dir, 'sos_dsw1.nc'))
sos_ghi.to_netcdf(os.path.join(save_dir, 'sos_ghi.nc'))
sos_dsw30.to_netcdf(os.path.join(save_dir, 'sos_dsw30.nc'))
# sos_dir.to_netcdf(os.path.join(save_dir, 'sos_dir.nc'))
# sos_dif.to_netcdf(os.path.join(save_dir, 'sos_dif.nc'))

In [8]:
# ONLY SAVE DIR AND DIF

def extract_site_data(site_key):
    x = final_utm_coords[site_key]['utm_x']
    y = final_utm_coords[site_key]['utm_y']
    
    site_ds = HRRR_solar.sel(x=x, y=y, method='nearest')[['dir', 'dif']]
    return site_ds.squeeze().compute()

# Extract for M1
m1_data = extract_site_data('m1_coords_utm')
m1_dir = m1_data['dir']
m1_dif = m1_data['dif']

save_dir = 'm1-dir-dif'
# os.makedirs(save_dir, exist_ok=True)

m1_dir.to_netcdf(os.path.join(save_dir, 'm1_dir_23.nc'))
m1_dif.to_netcdf(os.path.join(save_dir, 'm1_dif_23.nc'))

In [11]:
# Define helper function to extract and compute data at one site
def extract_site_data(site_key):
    x = final_utm_coords[site_key]['utm_x']
    y = final_utm_coords[site_key]['utm_y']
    
    site_ds = HRRR_solar.sel(x=x, y=y, method='nearest')[['ghi', 'dsw1', 'dsw3', 'dsw3h', 'k']]
    return site_ds.squeeze().compute()


# Extract for S3
s3_data = extract_site_data('s3_coords_utm')
s3_ghi = s3_data['ghi']
s3_dsw1 = s3_data['dsw1']
s3_dsw30 = s3_data['dsw3']
s3_dsw3 = s3_data['dsw3h']
s3_k = s3_data['k']

save_dir = 's3-new'
os.makedirs(save_dir, exist_ok=True)

s3_dsw3.to_netcdf(os.path.join(save_dir, 's3_dsw3_22.nc'))
s3_k.to_netcdf(os.path.join(save_dir, 's3_k_22.nc'))
s3_dsw1.to_netcdf(os.path.join(save_dir, 's3_dsw1_22.nc'))
s3_ghi.to_netcdf(os.path.join(save_dir, 's3_ghi_22.nc'))
s3_dsw30.to_netcdf(os.path.join(save_dir, 's3_dsw30_22.nc'))

In [12]:
client.shutdown()